# MEIKA Vox · Santa María\nRuta simple para transcribir audios privados desde Google Drive sin subirlos al repositorio.\n\n**Uso:** en Colab, activa GPU si está disponible y luego ejecuta todas las celdas.\n

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg git\n!python -m pip install -q --upgrade pip\n!rm -rf /content/meika-vox\n!git clone -q https://github.com/MeikaLab/meika-vox.git\n%cd /content/meika-vox\n!python -m pip install -q -e '.[whisperx]'\n!meika-vox doctor\n

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n

## Define tus audios\nCambia solo las dos rutas si tus archivos están en otra carpeta de Drive.\n

In [ ]:
from pathlib import Path\n\nAUDIO_LAS_CABRAS = Path('/content/drive/MyDrive/José Contreras 2.m4a')\nAUDIO_ESCUELA = Path('/content/drive/MyDrive/Grabación mesa 1.m4a')\nOUTPUT = Path('/content/drive/MyDrive/MEIKA_Vox/Santa_Maria_2026')\nOUTPUT.mkdir(parents=True, exist_ok=True)\n\nfor p in [AUDIO_LAS_CABRAS, AUDIO_ESCUELA]:\n    print(p, '✅' if p.exists() else '❌ NO ENCONTRADO')\n

## Primera pasada: ASR sin diarización\nAsí obtenemos texto primero sin bloquear por Hugging Face/pyannote.\n

In [ ]:
import subprocess\n\ndef transcribe(audio: Path):\n    cmd = [\n        'meika-vox','transcribe',str(audio),\n        '--project-id','SM26',\n        '--language','es',\n        '--model','large-v3',\n        '--glossary','configs/glossaries/santa_maria_2026.json',\n        '--output',str(OUTPUT),\n    ]\n    print(' '.join(cmd))\n    subprocess.run(cmd, check=True)\n\ntranscribe(AUDIO_LAS_CABRAS)\ntranscribe(AUDIO_ESCUELA)\n

## Ver los archivos generados\nLa salida queda en `MyDrive/MEIKA_Vox/Santa_Maria_2026/`.\n

In [ ]:
for p in OUTPUT.rglob('*'):\n    if p.is_file():\n        print(p.relative_to(OUTPUT))\n

Archivos principales: `transcript_normalized.txt`, `transcript_raw.jsonl`, `words.jsonl`, `speaker_turns.jsonl`, `qa.json` y `audio_qa.json`.\n\nLa diarización se agrega después; no es necesaria para validar primero la calidad de la transcripción.\n